# GPT

In [1]:
import os
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import triton
import triton.language as tl
import math
import time

In [2]:
device=torch.device('cuda:0')

In [3]:
# dataset

def dataset(url, filepath):
    if not os.path.exists(filepath):
        print(f"Downloading dataset from {url}...")
        response = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(response.content)
        print(f"Dataset downloaded and saved to {filepath}.")
    else:
        print(f"Dataset already exists at {filepath}.")


url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filepath = "input.txt"

dataset(url, filepath)
with open('input.txt', 'r') as f:
    text = f.read()

Dataset already exists at input.txt.


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 65


In [5]:
str2idx = { ch:i for i,ch in enumerate(chars) }
idx2str = { i:ch for i,ch in enumerate(chars) } 


def encode(text):
    return torch.tensor([str2idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join([idx2str[i.item()] for i in indices])

def train_test_split(data, device):
    n = int(0.9 * len(data))
    train_data = data[:n].to(device)
    test_data = data[n:].to(device)
    return train_data, test_data

data = encode(text)
device = torch.device('cuda:0')
train_data, test_data = train_test_split(data, device)

In [6]:
print(f"Training data size: {train_data.numel()} characters")
print(f"Testing data size: {test_data.numel()} characters")

Training data size: 1003854 characters
Testing data size: 111540 characters


# Triton kernels

In [18]:
@triton.jit
def softmax_kernel(
    output_ptr, input_ptr, input_row_stride, output_row_stride, n_cols,
    BLOCK_SIZE: tl.constexpr
):
    row_idx = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols

    input_row_ptr = input_ptr + row_idx * input_row_stride + col_offsets
    output_row_ptr = output_ptr + row_idx * output_row_stride + col_offsets

    logits = tl.load(input_row_ptr, mask=mask, other=float('-inf'))
    max_logits = tl.max(logits, axis=0)
    logits = logits - max_logits
    exp_logits = tl.exp(logits)
    sum_exp_logits = tl.sum(exp_logits, axis=0) + 1e-6

    softmax_output = exp_logits / sum_exp_logits
    tl.store(output_row_ptr, softmax_output, mask=mask)

@triton.jit
def layer_norm_kernel(
    x_ptr, weight_ptr, bias_ptr, y_ptr,
    N, eps: tl.constexpr,
    BLOCK_SIZE: tl.constexpr
):
    row_idx = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N

    x_offset = x_ptr + row_idx * N + cols
    x = tl.load(x_offset, mask=mask, other=0.0)

    mean = tl.sum(x, axis=0) / N
    x_centered = x - mean
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)

    w = tl.load(weight_ptr + cols, mask=mask, other=1.0)
    b = tl.load(bias_ptr + cols, mask=mask, other=0.0)

    y = (x_centered * rstd) * w + b
    tl.store(y_ptr + row_idx * N + cols, y, mask=mask)

@triton.jit
def cross_entropy_loss_kernel(
    logits_ptr, targets_ptr, loss_ptr, 
    n_classes, n_elements,
    BLOCK_SIZE: tl.constexpr
):
    pid = tl.program_id(0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    targets = tl.load(targets_ptr + offsets, mask=mask, other=-1)

    row_max = tl.full([BLOCK_SIZE], float('-inf'), dtype=tl.float32)
    row_sum = tl.zeros([BLOCK_SIZE], dtype=tl.float32)

    for i in range(n_classes):
        col_offset = offsets * n_classes + i
        logit = tl.load(logits_ptr + col_offset, mask=mask, other=float('-inf'))
        row_max = tl.maximum(row_max, logit)

    loss = tl.zeros([BLOCK_SIZE], dtype=tl.float32)
    for i in range(n_classes):
        col_offset = offsets * n_classes + i
        logit = tl.load(logits_ptr + col_offset, mask=mask, other=float('-inf'))
        exp_logit = tl.exp(logit - row_max)
        row_sum += exp_logit
        loss = tl.where(targets == i, loss - logit + row_max, loss)

    loss += tl.log(row_sum)

    tl.store(loss_ptr + offsets, loss, mask=mask)

@triton.jit
def gelu_kernel(
    x_ptr, y_ptr, n_elements,
    BLOCK_SIZE: tl.constexpr
):
    offsets = tl.program_id(0) * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask)

    sqrt_2_over_pi = 0.7978845608028654
    coeff = sqrt_2_over_pi * (1 + 0.044715 * x * x)
    y = 0.5 * x * (1 + (x * coeff) / (1 + tl.abs(x * coeff)))

    tl.store(y_ptr + offsets, y, mask=mask)

In [19]:
import torch.nn as nn


class TritonSoftmax(nn.Module):
    def forward(self, x):
        original_shape = x.shape
        if len(original_shape) > 2:
            x = x.view(-1, original_shape[-1])
        x = x.clamp(-100, 100)
        B, N = x.shape
        y = torch.empty_like(x)
        grid = lambda meta: (B,)
        softmax_kernel[grid](
            y, x,
            x.stride(0), y.stride(0), N,
            BLOCK_SIZE=triton.next_power_of_2(N)
        )
        y = y + 1e-8
        y = y / y.sum(dim=-1, keepdim=True)
        return y.view(original_shape)
    
def triton_cross_entropy_loss(logits, targets):
    return TritonCrossEntropyLoss.apply(logits, targets)

class TritonCrossEntropyLoss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, logits, targets):
        n_elements, n_classes = logits.shape
        loss = torch.empty(n_elements, device=logits.device, dtype=logits.dtype)
        
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        
        cross_entropy_loss_kernel[grid](
            logits, targets, loss,
            n_classes, n_elements,
            BLOCK_SIZE=1024
        )
        
        ctx.save_for_backward(logits, targets)
        return loss.mean()

    @staticmethod
    def backward(ctx, grad_output):
        logits, targets = ctx.saved_tensors
        batch_size, n_classes = logits.shape

        logits_exp = torch.exp(logits - logits.max(dim=-1, keepdim=True).values)
        softmax_output = logits_exp / logits_exp.sum(dim=-1, keepdim=True)

        grad_input = softmax_output.clone()
        grad_input.scatter_add_(1, targets.unsqueeze(1), -torch.ones_like(grad_input))
        grad_input *= grad_output.view(-1, 1) / batch_size

        return grad_input, None


class TritonLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.normalized_shape = tuple(normalized_shape) if isinstance(normalized_shape, (tuple, list)) else (normalized_shape,)
        self.weight = nn.Parameter(torch.ones(self.normalized_shape))
        self.bias = nn.Parameter(torch.zeros(self.normalized_shape))
        self.eps = eps

    def forward(self, x):
        assert x.shape[-len(self.normalized_shape):] == self.normalized_shape, "Input shape does not match normalized_shape."
        y = torch.empty_like(x)
        x_ = x.reshape(-1, self.normalized_shape[-1])
        y_ = y.reshape(-1, self.normalized_shape[-1])
        M, N = x_.shape
        grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE']),)
        layer_norm_kernel[grid](
            x_, self.weight, self.bias, y_,
            N, eps=self.eps,
            BLOCK_SIZE=128
        )
        return y

class TritonGELU(nn.Module):
    def forward(self, x):
        n_elements = x.numel()
        y = torch.empty_like(x)
        grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
        gelu_kernel[grid](
            x, y, n_elements,
            BLOCK_SIZE=1024
        )
        return y

In [20]:
import wandb


def train(model, train_data, val_data, batch_size, seq_length, learning_rate, num_epochs, wandb_project=None, wandb_run_name=None, timeout=None):
    
    if wandb_project:
        wandb.init(project=wandb_project, name=wandb_run_name, settings=wandb.Settings(init_timeout=timeout))
        wandb.config.update({
            "batch_size": batch_size,
            "seq_length": seq_length,
            "learning_rate": learning_rate,
            "num_epochs": num_epochs,
            "model_dim": model.dim,
            "num_heads": model.num_heads,
            "num_layers": model.num_layers
        })

    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    def get_batch(split):
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - seq_length, (batch_size,))
        x = torch.stack([data[i:i+seq_length] for i in ix])
        y = torch.stack([data[i+1:i+seq_length+1] for i in ix])
        return x.to(model.token_embedding.weight.device), y.to(model.token_embedding.weight.device)

    def estimate_mfu(model, dt):
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = sum(p.numel() for p in model.parameters())
        L, H, Q, T = model.num_layers, model.num_heads, model.dim // model.num_heads, model.seq_length
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T * batch_size
        flops_achieved = flops_per_fwdbwd * (1.0/dt) 
        flops_promised = 312e12 
        mfu = flops_achieved / flops_promised
        return mfu

    iter_num = 0
    best_val_loss = float('inf')
    val_losses = []

    model.train()
    t0 = time.time()
    for epoch in range(num_epochs):
        for _ in range(100): 
            iter_num += 1

            t_start = time.time()

            xb, yb = get_batch('train')
            t_data = time.time()

            logits = model(xb)
            t_forward = time.time()

            loss = model.compute_loss(logits, yb)
            t_loss = time.time()

            if torch.isnan(loss).any() or torch.isinf(loss).any():
                print(f"Warning: NaN or Inf detected in loss at iteration {iter_num}")
                print(f"Logits min: {logits.min()}, max: {logits.max()}")
                print(f"Target min: {yb.min()}, max: {yb.max()}")
                continue

            optimizer.zero_grad()
            loss.backward()
            t_backward = time.time()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            torch.cuda.synchronize()
            t_optim = time.time()

            if iter_num % 10 == 0:
                dt = t_optim - t_start
                dt_data = t_data - t_start
                dt_forward = t_forward - t_data
                dt_loss = t_loss - t_forward
                dt_backward = t_backward - t_loss
                dt_optim = t_optim - t_backward
                mfu = estimate_mfu(model, dt)
                
                print(f"iter {iter_num}: loss {loss.item():.4f}, time {dt*1000:.2f}ms, mfu {mfu*100:.2f}%")

                if wandb_project:
                    wandb.log({
                        "train/loss": loss.item(),
                        "iter": iter_num,
                        "mfu": mfu*100
                    })
                
        scheduler.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for _ in range(50): 
                xb, yb = get_batch('val')
                logits = model(xb)
                val_loss += model.compute_loss(logits, yb).item()
        val_loss /= 50
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}")


        if wandb_project:
            wandb.log({
                "val/loss": val_loss,
                "epoch": epoch+1,
                "learning_rate": scheduler.get_last_lr()[0]
            })

        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'Checkpoints/nanoGPT_cpkt.pth')
            print(f"Saved checkpoint for validation loss: {best_val_loss:.4f}")

        model.train()

    return model, val_losses

In [29]:
vocab_size = 512
dim = 384
num_heads = 8
num_layers = 8
seq_length = 512
dropout = 0.1
batch_size = 64
learning_rate = 3e-4
num_epochs = 500


model = NanoGPT(
    vocab_size=vocab_size,
    dim=dim,
    num_heads=num_heads,
    num_layers=num_layers,
    seq_length=seq_length,
    dropout=dropout
).to(device)

model.config = type('Config', (), {
    'n_layer': num_layers,
    'n_head': num_heads,
    'n_embd': dim,
    'block_size': seq_length
})

model, validation_losses = train(
    model,
    train_data,
    test_data,
    batch_size=batch_size,
    seq_length=seq_length,
    learning_rate=learning_rate,
    num_epochs=num_epochs,
    wandb_project=None,
    wandb_run_name=None,
    timeout=None
)

iter 10: loss 6.1776, time 194.00ms, mfu 5.82%
iter 20: loss 6.1163, time 193.69ms, mfu 5.83%
iter 30: loss 6.0566, time 193.93ms, mfu 5.82%
iter 40: loss 6.0013, time 193.94ms, mfu 5.82%
iter 50: loss 5.9451, time 193.79ms, mfu 5.82%
iter 60: loss 5.8958, time 193.98ms, mfu 5.82%
iter 70: loss 5.8443, time 367.77ms, mfu 3.07%
iter 80: loss 5.7995, time 194.26ms, mfu 5.81%
iter 90: loss 5.7573, time 194.45ms, mfu 5.80%
iter 100: loss 5.7178, time 194.35ms, mfu 5.81%
Epoch 1/500, Validation Loss: 6.2115
Saved checkpoint for validation loss: 6.2115
iter 110: loss 5.6795, time 336.28ms, mfu 3.36%
iter 120: loss 5.6486, time 274.71ms, mfu 4.11%
iter 130: loss 5.6201, time 194.59ms, mfu 5.80%
iter 140: loss 5.5849, time 194.38ms, mfu 5.80%
iter 150: loss 5.5550, time 393.74ms, mfu 2.87%
iter 160: loss 5.5254, time 194.35ms, mfu 5.81%
iter 170: loss 5.5010, time 370.71ms, mfu 3.04%
iter 180: loss 5.4872, time 210.52ms, mfu 5.36%
iter 190: loss 5.4685, time 211.20ms, mfu 5.34%
iter 200: loss 

KeyboardInterrupt: 

<All keys matched successfully>

In [23]:
model.eval()
start_text = "Once upon"

def inference(text: str):
    input_ids = encode(start_text).unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(240):
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :]
            next_token_logits = torch.clamp(next_token_logits, -100, 100)
            probs = F.softmax(next_token_logits, dim=-1) + 1e-8
            probs = probs / probs.sum()
            if torch.isnan(probs).any() or torch.isinf(probs).any():
                probs = torch.ones_like(probs) / probs.shape[-1]
            
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

    return decode(input_ids[0].cpu())